# 00 · Pipeline: collect, parse, clean, embed

Everything from `01` → `02` → `03` in one run, with the steps overlapping:

- **Collect** (`01`) runs in a background thread. It mostly waits on Steam's request limit, so it barely uses the computer.
- Meanwhile, every `EVERY_MINUTES` and once more when collection ends, what has arrived is **parsed and cleaned** (`02`) and the kept reviews are **embedded** on the GPU (`03`). All three only touch new data.

Interrupting the notebook stops collection too (after its current request); everything collected so far is kept, and the next run continues where it stopped. Topics are not part of this: refit or assign them once enough data is in.

In [ ]:
import logging

from src.collect.run import plan
from src.config import CONFIG
from src.db import read
from src.db.connection import init_db
from src.pipeline import run

logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
init_db()

In [ ]:
SOURCES = "all"       # or e.g. ["reviews"]; all = store_details, reviews, news, player_counts
APPIDS = None         # any Steam app IDs; None = all games in configs/games.yaml (new IDs are added there)
PER_LANG = 0          # reviews per game x language; None = config, 0 = all reviews
EVERY_MINUTES = 15    # how often new data is parsed, cleaned and embedded during collection
EMBED = True          # False: only parse and clean (e.g. no GPU)
MODEL = "BAAI/bge-m3"

jobs = plan(SOURCES, APPIDS, PER_LANG)
requests = jobs["requests"].sum()
hours = requests / CONFIG.http.host_rps["api.steampowered.com"] / 3600
print(f"{len(jobs)} jobs, {requests:,} requests, about {hours:.1f} h of collecting")
jobs

## Run

In [ ]:
result = run(jobs, MODEL, EVERY_MINUTES, EMBED)
result.groupby(["source", "status"]).size()

In [ ]:
# Failed jobs (re-run this notebook to retry them)
result[result["status"] == "failed"]

In [ ]:
read.review_collection()